# 3D U-Net + FADC-Bottleneck **V2.1** — Attention capacity bump

**Purpose:** Test whether the v2.1 capacity bump (avg+max concat, wider bottleneck, always-on filter_fc) lifts Dice at the strongest v1 placement (Bottleneck 0.6801). Direct A/B against **v2 buggy Bottleneck** (0.6648, killed the "spatial-fix helps everywhere" story on 2026-06-25) and against **v1 Bottleneck** (0.6801 s=42).

**What changed vs v2 (commits `02b869d` + `c8393a9` on `feature/fadc-3d-v2-spatial`):**
1. **Avg + max concat descriptor** — c_att / f_att / s_att now see both mean and peak statistics of the input feature map, not just avg.
2. **Wider attention bottleneck** — `reduction 0.0625 → 0.125`, `min_channel 16 → 32`. Attention FC has 2× more capacity to route per-input signal.
3. **Always-on filter attention** — the `in_planes == groups == out_planes` skip removed. `filter_fc` is now constructed at every FADC layer (was a no-op skip in this arch — behavior unchanged, but cleaner code).
4. **k_att temperature endpoint** — cosine anneals **4.0 → 0.8** (was 4.0 → 1.0) over full 100 epochs. Sharpens the softmax to force firmer per-voxel dilation commitment.
5. Unchanged from v2: spatial k_att path, warm bias init (0.5), FFT `freq_select`.

**Reference numbers to beat:**
- v1 Bottleneck s=42: **0.6801** (single-seed win, n=3 mean 0.6735 ± 0.0057)
- v1 Bottleneck+TTA n=3: **0.7010 ± 0.0046**
- v2 buggy Bottleneck s=42: 0.6648 (LOST v1 by −0.0153)
- v2 fixed Encoder s=42 at ep70: 0.6267 (plateaued — motivated v2.1)
- Baseline UNet3D uncontrolled: 0.6735

**Decision gate for Bottleneck v2.1 s=42:**
- Best Dice **≥ 0.70** → capacity bump rescued the strongest placement. Queue s=123 and s=999. Path A revival.
- Best Dice **0.68 – 0.70** → matches v1 Bottleneck. Capacity helps recover what temperature-scope bug broke, but no new win. One extra seed to disambiguate.
- Best Dice **0.66 – 0.68** → modest lift over v2 buggy (0.6648) but under v1. Consistent with `[[project_attention_failure_analysis]]` — capacity is not the primary bottleneck.
- Best Dice **< 0.66** → v2.1 fails to reach even baseline at Bottleneck. Path B (mechanism-first) locked; v2.1 documented as capacity ablation.

**Model:** UNet3DFADC_V2 with `fadc_placement='bottleneck'` — same depth/widths as v1, only the FADC block and attention module changed.
**Dataset:** MAMA-MIA (1200 train / 306 val), 2-channel input.
**Patch size:** 128×128×64 | **Epochs:** 100 | **Warmup:** 5 | **val_every:** 10
**k_att schedule:** T=4.0 → 0.8 cosine, 100 epochs, `set_temperature(t)` at start of every epoch.
**GPU:** Kaggle T4×2, cudnn.deterministic=True. ~10 h wall time.

**IMPORTANT:** Download `best_model.pth`, `train_log.json`, `meta.json` from `OUTPUT_DIR` at each val_every landing per `[[feedback_download_kaggle_checkpoints]]` — `/kaggle/working` wipes on session close.

In [ ]:
# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
SEED = 42  # <<< CHANGE TO 123 AND 999 FOR ADDITIONAL RUNS ONLY IF GATE PASSES

DATA_ROOT    = "/kaggle/input/datasets/bharathvemurik/mama-mia-preprocessed-cache-2ch"
OUTPUT_DIR   = f"/kaggle/working/outputs/fadc_bottleneck_v21_2ch_100ep_s{SEED}"
CODE_DIR     = "/kaggle/working/FADC-3D"

EPOCHS       = 100
BATCH_SIZE   = 2
NUM_WORKERS  = 4
PATCH_SIZE   = [128, 128, 64]
WARMUP       = 5
VAL_EVERY    = 10   # match Encoder v2.1 cadence for apples-to-apples trajectory

# V2-specific — temperature anneal for k_att softmax
K_ATT_TEMP_START = 4.0
K_ATT_TEMP_END   = 0.8
# Anneal over the whole training run
K_ATT_ANNEAL_EPOCHS = 100

RESUME_FROM  = ""
PREPROCESSED_CACHE_DIR = "/kaggle/input/datasets/bharathvemurik/mama-mia-preprocessed-cache-2ch"

# The new branch where v2 lives — change to 'main' once merged.
GIT_BRANCH = "feature/fadc-3d-v2-spatial"

print(f"SEED         : {SEED}")
print(f"OUTPUT_DIR   : {OUTPUT_DIR}")
print(f"GIT_BRANCH   : {GIT_BRANCH}")
print(f"k_att temp   : {K_ATT_TEMP_START} -> {K_ATT_TEMP_END} over {K_ATT_ANNEAL_EPOCHS} ep")
print(f"val_every    : {VAL_EVERY}")

In [ ]:
# ─────────────────────────────────────────────
# 1. INSTALL DEPENDENCIES
# ─────────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "monai",
                "--upgrade-strategy", "only-if-needed", "-q"], check=True)

import torch
print(f"PyTorch        : {torch.__version__}")
print(f"CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU            : {torch.cuda.get_device_name(0)}")

In [ ]:
# ─────────────────────────────────────────────
# 2. CLONE / UPDATE CODE FROM GITHUB — V2 BRANCH
# ─────────────────────────────────────────────
import os, sys

if os.path.exists(CODE_DIR):
    print("Repo exists — fetching + checking out v2 branch...")
    os.system(f"git -C {CODE_DIR} fetch --all")
    os.system(f"git -C {CODE_DIR} checkout {GIT_BRANCH}")
    os.system(f"git -C {CODE_DIR} pull")
else:
    os.system(f"git clone -b {GIT_BRANCH} https://github.com/Vemuri-BK/FADC-3D.git {CODE_DIR}")
    print("Repo cloned on v2 branch.")

sys.path.insert(0, CODE_DIR)

# Verify v2 modules exist
assert os.path.exists(os.path.join(CODE_DIR, "fadc_3d_v2", "omni_attention_3d_spatial.py")), \
    "fadc_3d_v2/omni_attention_3d_spatial.py missing — wrong branch?"
assert os.path.exists(os.path.join(CODE_DIR, "models", "unet_3d_fadc_v2.py")), \
    "models/unet_3d_fadc_v2.py missing — wrong branch?"
assert os.path.exists(os.path.join(CODE_DIR, "training", "train_centralized_v2.py")), \
    "training/train_centralized_v2.py missing — wrong branch?"
print("V2 modules present.")

In [ ]:
# ─────────────────────────────────────────────
# 3. SMOKE TEST — verify v2.1 model instantiates + forward + spatial k_att shape
#    + v2.1 fc.weight input dim doubled (avg+max concat)
# ─────────────────────────────────────────────
import torch
from models.unet_3d_fadc_v2 import UNet3DFADC_V2
from fadc_3d_v2.adaptive_dilated_conv_3d_v2 import AdaptiveDilatedConv3DV2
from fadc_3d_v2.omni_attention_3d_spatial import OmniAttention3DSpatial

device = torch.device("cuda")
model = UNet3DFADC_V2(in_channels=2, out_channels=2, base_filters=32,
                       fadc_placement='bottleneck').to(device).eval()
n_params = sum(p.numel() for p in model.parameters())
print(f"Params: {n_params:,}")

# v2.1: fc input dim must equal 2*in_planes (avg+max concat)
for name, m in model.named_modules():
    if isinstance(m, OmniAttention3DSpatial):
        in_planes  = m.channel_fc.out_channels
        fc_in      = m.fc.in_channels
        assert fc_in == 2 * in_planes, \
            f"v2.1 avg+max concat missing at {name}: fc.in={fc_in}, expected {2*in_planes}"
        assert m.filter_fc is not None, f"v2.1 filter_fc missing at {name}"
print("  [OK] v2.1 fc input dim doubled at every FADC layer (avg+max concat)")
print("  [OK] filter_fc present at every FADC layer")

# Verify k_att is spatial via a hook
shapes = []
def hook(_mod, _inp, out):
    shapes.append(out[3].shape)  # k_att is 4th tuple element
for m in model.modules():
    if isinstance(m, AdaptiveDilatedConv3DV2):
        m.omni_att.register_forward_hook(hook)

x = torch.randn(1, 2, *PATCH_SIZE).to(device)
with torch.no_grad():
    y = model(x)
print(f"Forward OK. Output: {y.shape}")
print(f"k_att shapes from each FADC block:")
for s in shapes:
    print(f"  {tuple(s)}  <-- has spatial dims (v1 would be (1, 2, 1, 1, 1))")
    assert len(s) == 5 and s[-3] > 1, "k_att is NOT spatial — wrong branch?"

del model, x, y
torch.cuda.empty_cache()
print("\nV2.1 smoke test PASSED.")

In [ ]:
# ─────────────────────────────────────────────
# 3. SMOKE TEST — verify v2 model instantiates + forward + spatial k_att shape
# ─────────────────────────────────────────────
import torch
from models.unet_3d_fadc_v2 import UNet3DFADC_V2
from fadc_3d_v2.adaptive_dilated_conv_3d_v2 import AdaptiveDilatedConv3DV2

device = torch.device("cuda")
model = UNet3DFADC_V2(in_channels=2, out_channels=2, base_filters=32,
                       fadc_placement='bottleneck').to(device).eval()
n_params = sum(p.numel() for p in model.parameters())
print(f"Params: {n_params:,}")

# Verify k_att is spatial via a hook
shapes = []
def hook(_mod, _inp, out):
    shapes.append(out[3].shape)  # k_att is 4th tuple element
for m in model.modules():
    if isinstance(m, AdaptiveDilatedConv3DV2):
        m.omni_att.register_forward_hook(hook)

x = torch.randn(1, 2, *PATCH_SIZE).to(device)
with torch.no_grad():
    y = model(x)
print(f"Forward OK. Output: {y.shape}")
print(f"k_att shapes from each FADC block:")
for s in shapes:
    print(f"  {tuple(s)}  <-- has spatial dims (v1 would be (1, 2, 1, 1, 1))")
    assert len(s) == 5 and s[-3] > 1, "k_att is NOT spatial — wrong branch?"

del model, x, y
torch.cuda.empty_cache()
print("\nV2 smoke test PASSED.")

In [ ]:
# ─────────────────────────────────────────────
# 5. LAUNCH V2.1 TRAINING — Bottleneck + avg+max + wider bottleneck + spatial k_att
# ─────────────────────────────────────────────
import os, subprocess, sys
os.makedirs(OUTPUT_DIR, exist_ok=True)

train_script = os.path.join(CODE_DIR, "training", "train_centralized_v2.py")

cmd = [
    sys.executable, "-u", train_script,
    "--model",          "unet3d_fadc_bottleneck_v2",
    "--data_root",      DATA_ROOT,
    "--output_dir",     OUTPUT_DIR,
    "--epochs",         str(EPOCHS),
    "--batch_size",     str(BATCH_SIZE),
    "--num_workers",    str(NUM_WORKERS),
    "--patch_size",     str(PATCH_SIZE[0]), str(PATCH_SIZE[1]), str(PATCH_SIZE[2]),
    "--warmup_epochs",  str(WARMUP),
    "--val_every",      str(VAL_EVERY),
    "--seed",           str(SEED),
    "--k_att_temp_start", str(K_ATT_TEMP_START),
    "--k_att_temp_end",   str(K_ATT_TEMP_END),
]
if K_ATT_ANNEAL_EPOCHS is not None:
    cmd += ["--k_att_anneal_epochs", str(K_ATT_ANNEAL_EPOCHS)]
if RESUME_FROM:
    cmd += ["--resume", RESUME_FROM]
if PREPROCESSED_CACHE_DIR:
    cmd += ["--preprocessed_cache_dir", PREPROCESSED_CACHE_DIR]

print("Command:\n  " + " ".join(cmd))
print("=" * 60, flush=True)

process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
while True:
    chunk = process.stdout.read(512)
    if not chunk:
        break
    sys.stdout.write(chunk.decode("utf-8", errors="replace"))
    sys.stdout.flush()
process.wait()
print(f"\nExit code: {process.returncode}")

In [ ]:
# ─────────────────────────────────────────────
# 5. LAUNCH V2 TRAINING — Bottleneck + spatial k_att + warm bias + temp anneal
# ─────────────────────────────────────────────
import os, subprocess, sys
os.makedirs(OUTPUT_DIR, exist_ok=True)

train_script = os.path.join(CODE_DIR, "training", "train_centralized_v2.py")

cmd = [
    sys.executable, "-u", train_script,
    "--model",          "unet3d_fadc_bottleneck_v2",
    "--data_root",      DATA_ROOT,
    "--output_dir",     OUTPUT_DIR,
    "--epochs",         str(EPOCHS),
    "--batch_size",     str(BATCH_SIZE),
    "--num_workers",    str(NUM_WORKERS),
    "--patch_size",     str(PATCH_SIZE[0]), str(PATCH_SIZE[1]), str(PATCH_SIZE[2]),
    "--warmup_epochs",  str(WARMUP),
    "--seed",           str(SEED),
    "--k_att_temp_start", str(K_ATT_TEMP_START),
    "--k_att_temp_end",   str(K_ATT_TEMP_END),
]
if K_ATT_ANNEAL_EPOCHS is not None:
    cmd += ["--k_att_anneal_epochs", str(K_ATT_ANNEAL_EPOCHS)]
if RESUME_FROM:
    cmd += ["--resume", RESUME_FROM]
if PREPROCESSED_CACHE_DIR:
    cmd += ["--preprocessed_cache_dir", PREPROCESSED_CACHE_DIR]

print("Command:\n  " + " ".join(cmd))
print("=" * 60, flush=True)

process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
while True:
    chunk = process.stdout.read(512)
    if not chunk:
        break
    sys.stdout.write(chunk.decode("utf-8", errors="replace"))
    sys.stdout.flush()
process.wait()
print(f"\nExit code: {process.returncode}")

In [ ]:
# ─────────────────────────────────────────────
# 7. POST-TRAIN k_att DIAGNOSTIC — did spatial k_att actually adapt?
#    Loads the best_model.pth, runs forward on a handful of val cases,
#    reports k_att std across cases per FADC block. Std > 0.005 = per-input
#    adaptation present.
# ─────────────────────────────────────────────
import os, sys, torch, numpy as np
from pathlib import Path
sys.path.insert(0, CODE_DIR)
from data.mama_mia_dataset import build_centralized_loaders
from models.unet_3d_fadc_v2 import UNet3DFADC_V2
from fadc_3d_v2.adaptive_dilated_conv_3d_v2 import AdaptiveDilatedConv3DV2

ckpt_path = os.path.join(OUTPUT_DIR, "best_model.pth")
if not os.path.exists(ckpt_path):
    print("No best_model.pth yet — run training first.")
else:
    device = torch.device("cuda")
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    cfg = ckpt["config"]
    model = UNet3DFADC_V2(in_channels=cfg["model"]["in_channels"],
                           out_channels=cfg["model"]["out_channels"],
                           base_filters=cfg["model"]["base_filters"],
                           fadc_placement='bottleneck').to(device).eval()
    model.load_state_dict(ckpt["model"])
    print(f"Loaded v2 ckpt: best_dice={ckpt['best_dice']:.4f}")

    captured = {}
    for name, mod in model.named_modules():
        if isinstance(mod, AdaptiveDilatedConv3DV2):
            captured[name] = []
            def mk(nm):
                def hook(_m, _i, out):
                    # out[3] is k_att, shape (b, n_branches, d, h, w) in v2
                    k = out[3].detach().cpu().numpy()
                    captured[nm].append(k[0].mean(axis=(1,2,3)))  # mean across voxels per branch
                return hook
            mod.omni_att.register_forward_hook(mk(name))
    print(f"Hooked {len(captured)} FADC blocks")

    # Run on 6 val cases
    split_csv = os.path.join(DATA_ROOT, "train_test_splits.csv")
    _, val_loader = build_centralized_loaders(
        data_root=DATA_ROOT,
        split_csv=split_csv if os.path.exists(split_csv) else None,
        cache_rate=0.0, num_workers=0, batch_size=1,
        preprocessed_cache_dir=PREPROCESSED_CACHE_DIR,
        patch_size=tuple(cfg["data"]["patch_size"]),
    )
    from monai.inferers import sliding_window_inference
    n = 0
    with torch.no_grad():
        for batch in val_loader:
            if n >= 6: break
            x = batch["image"].to(device)
            _ = sliding_window_inference(x, tuple(cfg["data"]["patch_size"]), 1, model, overlap=0.0)
            n += 1
    print(f"\nProcessed {n} val cases")
    print("=" * 78)
    print("V2 k_att per-input adaptation diagnostic")
    print("=" * 78)
    print(f"{'block':<30}{'k(d=1) std':>14}{'k(d=2) std':>14}{'adapt?':>10}")
    for nm in captured:
        arr = np.array(captured[nm])  # (n_forwards * patches, n_branches)
        if arr.size == 0:
            continue
        k1_std = float(arr[:, 0].std(ddof=1)) if len(arr) > 1 else 0.0
        k2_std = float(arr[:, 1].std(ddof=1)) if len(arr) > 1 else 0.0
        adapts = (k1_std > 0.005) or (k2_std > 0.005)
        print(f"{nm:<30}{k1_std:>14.4f}{k2_std:>14.4f}{'YES' if adapts else 'no':>10}")
    print("\nCompare with v1 Bottleneck s=42 diagnostic (2026-06-25):")
    print("  bottleneck.conv1  k(d=1) std=0.0005  -> no adaptation")
    print("  bottleneck.conv2  k(d=1) std=0.0001  -> no adaptation")

In [ ]:
# ─────────────────────────────────────────────
# 7. POST-TRAIN k_att DIAGNOSTIC — did spatial k_att actually adapt?
#    Loads the best_model.pth, runs forward on a handful of val cases,
#    reports k_att std across cases per FADC block. Std > 0.005 = per-input
#    adaptation present.
# ─────────────────────────────────────────────
import os, sys, torch, numpy as np
from pathlib import Path
sys.path.insert(0, CODE_DIR)
from data.mama_mia_dataset import build_centralized_loaders
from models.unet_3d_fadc_v2 import UNet3DFADC_V2
from fadc_3d_v2.adaptive_dilated_conv_3d_v2 import AdaptiveDilatedConv3DV2

ckpt_path = os.path.join(OUTPUT_DIR, "best_model.pth")
if not os.path.exists(ckpt_path):
    print("No best_model.pth yet — run training first.")
else:
    device = torch.device("cuda")
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    cfg = ckpt["config"]
    model = UNet3DFADC_V2(in_channels=cfg["model"]["in_channels"],
                           out_channels=cfg["model"]["out_channels"],
                           base_filters=cfg["model"]["base_filters"],
                           fadc_placement='bottleneck').to(device).eval()
    model.load_state_dict(ckpt["model"])
    print(f"Loaded v2 ckpt: best_dice={ckpt['best_dice']:.4f}")

    captured = {}
    for name, mod in model.named_modules():
        if isinstance(mod, AdaptiveDilatedConv3DV2):
            captured[name] = []
            def mk(nm):
                def hook(_m, _i, out):
                    # out[3] is k_att, shape (b, n_branches, d, h, w) in v2
                    k = out[3].detach().cpu().numpy()
                    captured[nm].append(k[0].mean(axis=(1,2,3)))  # mean across voxels per branch
                return hook
            mod.omni_att.register_forward_hook(mk(name))
    print(f"Hooked {len(captured)} FADC blocks")

    # Run on 6 val cases
    split_csv = os.path.join(DATA_ROOT, "train_test_splits.csv")
    _, val_loader = build_centralized_loaders(
        data_root=DATA_ROOT,
        split_csv=split_csv if os.path.exists(split_csv) else None,
        cache_rate=0.0, num_workers=0, batch_size=1,
        preprocessed_cache_dir=PREPROCESSED_CACHE_DIR,
        patch_size=tuple(cfg["data"]["patch_size"]),
    )
    from monai.inferers import sliding_window_inference
    n = 0
    with torch.no_grad():
        for batch in val_loader:
            if n >= 6: break
            x = batch["image"].to(device)
            for nm in captured: captured[nm].append(None) if False else None
            _ = sliding_window_inference(x, tuple(cfg["data"]["patch_size"]), 1, model, overlap=0.0)
            n += 1
    print(f"\nProcessed {n} val cases")
    print("=" * 78)
    print("V2 k_att per-input adaptation diagnostic")
    print("=" * 78)
    print(f"{'block':<30}{'k(d=1) std':>14}{'k(d=2) std':>14}{'adapt?':>10}")
    for nm in captured:
        arr = np.array(captured[nm])  # (n_forwards * patches, n_branches)
        if arr.size == 0:
            continue
        k1_std = float(arr[:, 0].std(ddof=1)) if len(arr) > 1 else 0.0
        k2_std = float(arr[:, 1].std(ddof=1)) if len(arr) > 1 else 0.0
        adapts = (k1_std > 0.005) or (k2_std > 0.005)
        print(f"{nm:<30}{k1_std:>14.4f}{k2_std:>14.4f}{'YES' if adapts else 'no':>10}")
    print("\nCompare with v1 Bottleneck s=42 diagnostic (2026-06-25):")
    print("  bottleneck.conv1  k(d=1) std=0.0005  -> no adaptation")
    print("  bottleneck.conv2  k(d=1) std=0.0001  -> no adaptation")

In [ ]:
# ─────────────────────────────────────────────
# 8. DOWNLOAD LINKS — grab everything before session closes
# ─────────────────────────────────────────────
import os
from IPython.display import FileLink, display
for fname in ("best_model.pth", "latest_checkpoint.pth", "train_log.json", "meta.json", "training_curves.png"):
    p = os.path.join(OUTPUT_DIR, fname)
    if os.path.exists(p):
        print(fname); display(FileLink(p))
    else:
        print(f"(missing) {fname}")